In [1]:
import os, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
# import earthpy as et
from functools import partial
import networkx as nx
# from tqdm import tqdm
from scipy import io
from hyperopt import fmin, tpe, hp, SparkTrials, STATUS_OK, Trials
# import mlflow
import project_path
from sklearn.neighbors import kneighbors_graph, radius_neighbors_graph
from src.util.t2m import t2m
from src.algos.lr_stss import lr_stss

In [2]:
# Load pre-processed Data
path = "C:\\Users"
path = os.path.join(path,'merti','datasets','nyc_taxi')

shapefile_path = 'C:\\Users\\merti\\datasets\\nyc_taxi\\taxi_zones_shapefile\\'
shapefile = os.path.join(shapefile_path,'taxi_zones.shp')
zones = gpd.read_file(shapefile)

zone_lookup = os.path.join(path, 'taxi_zone_lookup.csv')
zone_lookup = pd.read_csv(zone_lookup)

pickups = np.load(os.path.join(path,"hourly_pickup.npy"))
dropoffs = np.load(os.path.join(path,"hourly_dropoff.npy"))
# Load Emre's settings
dates = io.loadmat('C:\\Users\\merti\\ResearchRepos\\TensorAnomalyDetection\\data\\nyc_taxi_data\\dates.mat')
regions = io.loadmat('C:\\Users\\merti\\ResearchRepos\\TensorAnomalyDetection\\data\\nyc_taxi_data\\regions.mat')
neighbors = io.loadmat('C:\\Users\\merti\\ResearchRepos\\TensorAnomalyDetection\\data\\nyc_taxi_data\\neighbors.mat')
regions=regions['regions'].ravel()

# Filter the data
pickups = pickups[regions-1, ...]
dropoffs = dropoffs[regions-1, ...]
zones = zones.iloc[regions-1]

In [3]:

pos = np.zeros((81,2))
pos[:,0] = zones.geometry.centroid.x.values
pos[:,1] = zones.geometry.centroid.y.values

position ={}
G_nyc = nx.Graph()
G_nyc.add_nodes_from([(regions[i], {'pos': pos[i,:], 'LocationID': regions[i], 'zone': zones.iloc[i]['zone']}) for i in range(81)])
edge_list =[]
for i in range(len(neighbors['neighbors'].ravel())):
    for neighbor in neighbors['neighbors'].ravel()[i].ravel():
        if np.any(np.isin(neighbor, regions)) and (neighbor!=regions[i]):
            edge_list.append((regions[i], neighbor))
    # edge_list = edge_list + [(regions[i], neighbor) for neighbor in neighbors['neighbors'].ravel()[i]]
    position[regions[i]]=pos[i,:]
G_nyc.add_edges_from(edge_list)
# G_nyc.nodes()
A = nx.adjacency_matrix(G_nyc).toarray()
Deg = np.diag(np.asarray(np.sum(A,axis=1)).ravel())
Dsq = np.linalg.inv(np.sqrt(Deg))
An = Dsq@A@Dsq

C:\Users\merti\AppData\Local\Temp\ipykernel_21016\3678542241.py:17: FutureWarning: adjacency_matrix will return a scipy.sparse array instead of a matrix in Networkx 3.0.
  A = nx.adjacency_matrix(G_nyc).toarray()


In [5]:
def detect_topk_events(anomaly_scores, ratio):
    events_start_ts = pd.to_datetime(['01-Jan-2018', '03-Jan-2018 16:00:00', '14-Jan-2018 09:00:00', '20-Jan-2018 08:00:00', 
                                    '28-Jan-2018 16:00:00', '04-Mar-2018 15:00:00', '31-Mar-2018 13:00:00', '17-Mar-2018 11:00:00',
                                    '20-Mar-2018 10:00:00', '21-Mar-2018 16:00:00', '01-Jul-2018 17:00:00', '04-Jul-2018 17:00:00',
                                    '25-Sep-2018 10:00:00', '04-Oct-2018 08:00:00', '04-Nov-2018 12:00:00', '09-Nov-2018 19:00:00',
                                    '22-Nov-2018 21:00:00', '04-Dec-2018 19:00:00', '16-Dec-2018 10:00:00', '31-Dec-2018 21:00:00'
                                    ])

    events_end_ts = pd.to_datetime(['01-Jan-2018 02:00:00', '03-Jan-2018 22:00:00', '14-Jan-2018 17:00:00', '20-Jan-2018 15:00:00',
                                '28-Jan-2018 23:00:00', '04-Mar-2018 22:00:00', '31-Mar-2018 20:00:00', '17-Mar-2018 17:00:00',
                                '20-Mar-2018 20:00:00', '21-Mar-2018 22:00:00', '01-Jul-2018 22:00:00', '04-Jul-2018 23:00:00',
                                '25-Sep-2018 20:00:00', '04-Oct-2018 15:00:00', '04-Nov-2018 17:00:00', '09-Nov-2018 23:30:00',
                                '22-Nov-2018 23:59:00', '04-Dec-2018 23:59:00', '16-Dec-2018 15:00:00', '31-Dec-2018 23:59:00'
    ])
    indd = np.flip(np.argsort(anomaly_scores, axis=None))
    ind = np.unravel_index(indd[:int(len(indd)*ratio)], anomaly_scores.shape)
    topk_event_idx = ind
    anomaly_mask = np.zeros(anomaly_scores.shape, dtype=bool)
    anomaly_mask[topk_event_idx] =1
    num_detected_events = 0
    idxs = np.arange(81)
    w = events_start_ts.week
    d = events_start_ts.day_of_week
    h_s = events_start_ts.hour
    h_e = events_end_ts.hour
    for i in range(20):
        event_mask = np.zeros(anomaly_scores.shape, dtype=bool)
        locations = dates['dates'][2].ravel()[i].ravel()
        
        for loc in locations: 
            event_mask[idxs[regions==loc], w[i]-1, d[i], h_s[i]:h_e[i]] = 1
            event_mask[idxs[regions==loc], w[i]-1, d[i], h_e[i]] = 1
        if np.any(event_mask & anomaly_mask):
            num_detected_events +=1
    return num_detected_events

In [6]:
## Control Variables
time_m = 2
local_m = 1
lda_2 = 100
psi = 1
maxit = 90
maxit2 = 40
psi_dist = np.array([0.65499501, 1, 0.13491694, 0.42404973])
## Independent variables
### Hyperparameters
search_space = {
    'psi': hp.loguniform('psi',0,3),
    'lda_1': hp.loguniform('lda_1',-1,3),
    'lda_t': hp.loguniform('lda_t',-1,3),
    'lda_l': hp.loguniform('lda_l',-1,3),
}
Deg = np.diag(np.asarray(np.sum(A,axis=1)).ravel())
Dsq = np.linalg.inv(np.sqrt(Deg))
An = Dsq@A@Dsq


def objective(inputs, pickups, An):
    ts = time.time()
#     def run_lrstss(pickups, inputs, An):
    Y = np.ma.masked_array(pickups, mask=np.zeros(pickups.shape,dtype=bool))
    psi = inputs['psi']
    res = lr_stss(Y, An, time_m,local_m, verbose=0, max_it2=maxit2, max_it=maxit,
        lda2=lda_2, lda1=inputs['lda_1'], lda_t=inputs['lda_t'],
        lda_loc=inputs['lda_l'], psis=psi_dist*psi, rho=0.1, rho_upd=-1)
#         return res
    
#     future = client.submit(run_lrstss, pickups, inputs, An)
#     res = future.result()
    abs_s = np.abs(res['S'])
    ratios = np.array([0.014, 0.07, 0.14, 0.3, 0.7, 1, 2, 3])/100
    num_detected_events = np.array([detect_topk_events(abs_s, r) for r in ratios])
    result = {'loss': -np.sum(num_detected_events),
              'topk_detection_sum': np.sum(num_detected_events),
              'num_detected_events': num_detected_events,
              'lda_1': inputs['lda_1'],
              'lda_2': lda_2,
              'lda_l': inputs['lda_l'],
              'lda_t': inputs['lda_t'],
              'psi': inputs['psi'],
              'maxit': maxit, 'maxit2': maxit2,
              'it': res['it'],
              'status': STATUS_OK,
              'eval_time': time.time()-ts}
    return result

In [4]:
trial_run = SparkTrials(parallelism=10)

RuntimeError: Java gateway process exited before sending its port number

In [ ]:
tuning = fmin(fn= partial(objective,
                        pickups=pickups, An=An),
                        space=search_space, algo=tpe.suggest,
                        max_evals=50,
                        trials=trial_run)